In [4]:
import pandas as pd
import numpy as np

def calculate_aerosol_exposure_risk(row):
    """
    Calculates cumulative toxicological risk scores for media-audited cosmetic formulations.
    Applies a strict 2.2 Risk Multiplier if aerosol delivery triggers tissue occlusion bounds.
    """
    try:
        # Standardize and isolate updated product and delivery columns
        product_name = str(row.get('product_name', '')).strip().lower()
        delivery_mechanism = str(row.get('delivery_mechanism', '')).strip().lower()

        # Extract numeric formulation concentration bounds
        base_concentration = float(row.get('formulation_concentration', 0.0))
        absorption_index = float(row.get('tissue_absorption_index', 1.0))

        # Null-guard and bounds validation to insulate database corruption
        if pd.isna(base_concentration) or base_concentration <= 0.0:
            return 0.0

        # Target compound logic loop (Scanning real product matrices for silicone vectors)
        # Identifies setting sprays, freeze sprays, sunscreens, and body oils
        if any(keyword in product_name for keyword in ['setting', 'spray', 'freeze', 'sunscreen', 'sunblock', 'oil']):
            if 'aerosol' in delivery_mechanism:
                # Apply high-vulnerability multi-factor risk multiplier for pressurized delivery
                calculated_score = (base_concentration * absorption_index) * 2.2
                return round(calculated_score, 4)

        # Default safety baseline calculation for standard mechanical pump sprays
        return round(base_concentration * absorption_index, 4)

    except (ValueError, TypeError) as system_error:
        # Prevent silent database corruption by routing exceptions to error logging channels
        print(f"[PIPELINE RUNTIME EXCEPTION] Ingestion failure on record row: {system_error}")
        return -1.0

In [6]:
# 1. Load the downloaded CSV data file into a Pandas DataFrame
df = pd.read_csv('aerosol_tracking.csv')

# 2. Run the risk calculation function row-by-row across the table
df['dynamic_exposure_score'] = df.apply(calculate_aerosol_exposure_risk, axis=1)

# 3. Sort the dataset from highest risk to lowest risk
df_sorted = df.sort_values(by='dynamic_exposure_score', ascending=False)

# 4. Display the fully populated table on the screen!
df_sorted

,product_id,brand_name,product_name,delivery_mechanism,formulation_concentration,tissue_absorption_index,episode_source,timeline_marker,dynamic_exposure_score
3,REAL-S60-04,got2b,Glued Blasting Freeze Spray,Aerosol,16.0,0.90,Season 6 Episodes,Wig/Edge Maintenance Daytime Prep,31.6800
0,REAL-S67-01,One/Size,On 'Til Dawn Mattifying Waterproof Setting Spray,Aerosol,15.5,0.92,Season 6 & 7 Episodes,Everyday Glam Room & Fire Pit Eliminations,31.3720
2,REAL-S67-03,Matrix,Vavoom Freezing Spray,Aerosol,14.8,0.91,Season 6 & 7 Episodes,Evening Styling & Recoupling Prep,29.6296
1,REAL-S67-02,One/Size,Oil Sucker Liquid Blotting Paper Touch-Up Spray,Aerosol,12.0,0.88,Season 6 & 7 Episodes,Mid-Filming Touch-Ups & Crew Maintenance,23.2320
4,REAL-S67-05,Sun Bum,Original SPF 50 Sunscreen Spray,Aerosol,10.0,0.90,Season 6 & 7 Episodes,Poolside Daytime Activities & Outdoor Challenges,19.8000
5,REAL-S67-06,Coola,Organic Sunscreen SPF 50 Sunblock Spray,Aerosol,8.5,0.95,Season 6 & 7 Episodes,Host Ariana Madix Outdoor Filming Blocks,17.7650
6,REAL-S70-07,Five Below x Love Island Merch,Spray & Slay Body Oil (Fiji Punch / Villa Vani...,Spray,14.0,0.38,Season 7 Episodes,Nighttime Vanity Prep & High-Reflective Camera...,5.3200
